In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import polars as pl
from preprocess_dataset_for_training import create_training_data
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd
import copy

In [2]:
# -------------------------------
# Reproducibility setup
# -------------------------------
seed = 42  
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
# torch.cuda.manual_seed_all(seed)  # if using multi-GPU

# For deterministic behavior (slower but exact reproducibility)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
# -------------------------------
# Device configuration
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# -------------------------------
# Load and preprocess data
# -------------------------------
file = "joined_df_with_weather"
joined_df = pl.read_parquet(f"./data/{file}.parquet")
X, y = create_training_data(joined_df)

In [5]:
X_np = X.to_numpy()
y_np = y.to_numpy()  # shape: (samples, 15 targets)
del joined_df, X
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, shuffle=False)
del X_np, y_np

# Feature scaling
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

# Target scaling (important for stable float32 regression)
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

# Convert to PyTorch tensors on CPU (we'll move batches to GPU)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)
del X_train, X_test, y_train, y_test

# -------------------------------
# DataLoaders for batching
# -------------------------------
batch_size = 256  
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

test_dataset = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# -------------------------------
# Define multi-output NN
# -------------------------------
class MultiOutputNN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_train_t.shape[1]
output_dim = y_train_t.shape[1]


def smape(y_true, y_pred, eps=1e-8):
    denom = (np.abs(y_true) + np.abs(y_pred)) + eps
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom, axis=0)

num_seeds = 100
num_targets = output_dim  # from existing cell
batch_size = 256

# containers
mae_all = np.zeros((num_seeds, num_targets))
rmse_all = np.zeros((num_seeds, num_targets))
r2_all = np.zeros((num_seeds, num_targets))
smape_all = np.zeros((num_seeds, num_targets))

# keep datasets (tensors on CPU) created previously: train_dataset, test_dataset
for s in range(num_seeds):
    seed = s
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    # recreate model & optimizer
    model_s = MultiOutputNN(input_dim, output_dim).to(device)
    optimizer_s = optim.Adam(model_s.parameters(), lr=1e-4)
    
    
    criterion_s = nn.HuberLoss(delta=1.0, reduction='mean') # Was MSE before
    

    # reproducible shuffling for DataLoader
    gen = torch.Generator()
    gen.manual_seed(seed)
    train_loader_s = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=gen)
    test_loader_s = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # early stopping
    best_val_loss = float('inf')
    counter = 0
    patience = 5
    best_state = copy.deepcopy(model_s.state_dict())

    epochs = 100
    for epoch in range(1, epochs + 1):
        model_s.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader_s:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer_s.zero_grad()
            outputs = model_s(X_batch)
            loss = criterion_s(outputs, y_batch)
            loss.backward()
            optimizer_s.step()

            running_loss += loss.item() * X_batch.size(0)
        train_loss = running_loss / len(train_loader_s.dataset)

        # validation
        model_s.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader_s:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                outputs = model_s(X_batch)
                val_loss_total += criterion_s(outputs, y_batch).item() * X_batch.size(0)
        val_loss = val_loss_total / len(test_loader_s.dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
            best_state = copy.deepcopy(model_s.state_dict())
        else:
            counter += 1
            if counter >= patience:
                break

    # load best model and evaluate
    model_s.load_state_dict(best_state)
    model_s.eval()
    y_pred_list = []
    y_true_list = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader_s:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model_s(X_batch)
            y_pred_list.append(outputs.cpu())
            y_true_list.append(y_batch.cpu())

    y_pred_scaled = torch.cat(y_pred_list).numpy()
    y_true_scaled = torch.cat(y_true_list).numpy()

    # inverse transform
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    y_true = scaler_y.inverse_transform(y_true_scaled)

    # compute metrics per target
    for i in range(num_targets):
        mae_all[s, i] = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse_all[s, i] = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2_all[s, i] = r2_score(y_true[:, i], y_pred[:, i])
    smape_all[s, :] = smape(y_true, y_pred)



In [6]:
# aggregate: mean across seeds (per-target)
mae_mean_per_target = mae_all.mean(axis=0)
rmse_mean_per_target = rmse_all.mean(axis=0)
r2_mean_per_target = r2_all.mean(axis=0)
smape_mean_per_target = smape_all.mean(axis=0)

# also overall averages across targets
overall = {
    "MAE": mae_mean_per_target.mean(),
    "RMSE": rmse_mean_per_target.mean(),
    "R2": r2_mean_per_target.mean(),
    "SMAPE": smape_mean_per_target.mean()
}

# prepare DataFrame per target
target_names = list(y.columns) if hasattr(y, "columns") else [f"target_{i}" for i in range(num_targets)]
metrics_df = pd.DataFrame({
    "Target": target_names,
    "MAE": mae_mean_per_target,
    "RMSE": rmse_mean_per_target,
    "R2": r2_mean_per_target,
    "SMAPE": smape_mean_per_target
})

display(metrics_df)
print("Overall averages across targets (mean over seeds then targets):")
print(overall)

# optional: save
metrics_df.to_csv(f"NN_{num_seeds}_seeds_Huber_{file}.csv", index=False)


,Target,MAE,RMSE,R2,SMAPE
0,Negative Balancing Energy Unit Price for Balan...,37.737499,69.219513,0.485084,112.230244
1,Positive Balancing Energy Unit Price for Balan...,37.639479,69.100625,0.486856,112.187778
2,System Direction (kWh)_t+15min,11500.627109,16768.857470,0.625585,109.160671
3,Negative Balancing Energy Unit Price for Balan...,43.443044,82.337901,0.271486,119.231105
4,Positive Balancing Energy Unit Price for Balan...,43.372141,82.231459,0.273368,119.320143
5,System Direction (kWh)_t+30min,13182.097637,19503.942111,0.493720,117.239832
6,Negative Balancing Energy Unit Price for Balan...,46.326600,87.265835,0.181693,123.674593
7,Positive Balancing Energy Unit Price for Balan...,46.217265,87.196072,0.183002,123.614159
8,System Direction (kWh)_t+45min,14225.169053,21358.039436,0.392963,121.916745
9,Negative Balancing Energy Unit Price for Balan...,48.191049,89.472125,0.139804,126.699216


Overall averages across targets (mean over seeds then targets):
{'MAE': np.float64(4677.018499303183), 'RMSE': np.float64(7043.0414639697065), 'R2': np.float64(0.29077672282854716), 'SMAPE': np.float64(121.80157356262205)}


In [7]:
# compute std across seeds (per-target)
mae_mean_per_target = mae_all.mean(axis=0)
mae_std_per_target = mae_all.std(axis=0)

rmse_mean_per_target = rmse_all.mean(axis=0)
rmse_std_per_target = rmse_all.std(axis=0)

r2_mean_per_target = r2_all.mean(axis=0)
r2_std_per_target = r2_all.std(axis=0)

smape_mean_per_target = smape_all.mean(axis=0)
smape_std_per_target = smape_all.std(axis=0)

# add std columns to existing metrics_df (keeps previous mean columns)
metrics_df["MAE_STD"] = mae_std_per_target
metrics_df["RMSE_STD"] = rmse_std_per_target
metrics_df["R2_STD"] = r2_std_per_target
metrics_df["SMAPE_STD"] = smape_std_per_target

# display table and print concise mean ± std per target
display(metrics_df)

for i, t in enumerate(target_names):
    print(
        f"{t}: MAE {mae_mean_per_target[i]:.3f} ± {mae_std_per_target[i]:.3f}, "
        f"RMSE {rmse_mean_per_target[i]:.3f} ± {rmse_std_per_target[i]:.3f}, "
        f"R2 {r2_mean_per_target[i]:.3f} ± {r2_std_per_target[i]:.3f}, "
        f"SMAPE {smape_mean_per_target[i]:.3f} ± {smape_std_per_target[i]:.3f}"
    )

,Target,MAE,RMSE,R2,SMAPE,MAE_STD,RMSE_STD,R2_STD,SMAPE_STD
0,Negative Balancing Energy Unit Price for Balan...,42.317448,71.943148,0.443704,116.557325,1.325625,1.224056,0.019015,1.459721
1,Positive Balancing Energy Unit Price for Balan...,42.255555,71.813249,0.445742,116.478491,1.259329,1.099104,0.017046,1.297343
2,System Direction (kWh)_t+15min,11776.149326,17075.893528,0.611661,110.389512,357.454388,498.174325,0.023000,1.939854
3,Negative Balancing Energy Unit Price for Balan...,48.535572,83.752832,0.246176,122.555146,1.414669,0.923770,0.016690,1.068081
4,Positive Balancing Energy Unit Price for Balan...,48.482166,83.634403,0.248301,122.550966,1.467852,0.949199,0.017142,1.018202
5,System Direction (kWh)_t+30min,13378.157207,19691.836352,0.483877,117.737555,241.680228,351.406698,0.018598,1.445245
6,Negative Balancing Energy Unit Price for Balan...,51.957812,88.497848,0.158329,126.107542,1.714834,1.074165,0.020566,0.972219
7,Positive Balancing Energy Unit Price for Balan...,51.931567,88.397306,0.160254,126.098485,1.553307,1.011651,0.019347,0.949546
8,System Direction (kWh)_t+45min,14423.463760,21529.691470,0.383132,122.152565,257.802582,318.178215,0.018367,1.239424
9,Negative Balancing Energy Unit Price for Balan...,54.348050,90.891951,0.112190,128.705211,1.576262,1.074863,0.021074,0.926788


Negative Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+15min: MAE 42.317 ± 1.326, RMSE 71.943 ± 1.224, R2 0.444 ± 0.019, SMAPE 116.557 ± 1.460
Positive Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+15min: MAE 42.256 ± 1.259, RMSE 71.813 ± 1.099, R2 0.446 ± 0.017, SMAPE 116.478 ± 1.297
System Direction (kWh)_t+15min: MAE 11776.149 ± 357.454, RMSE 17075.894 ± 498.174, R2 0.612 ± 0.023, SMAPE 110.390 ± 1.940
Negative Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+30min: MAE 48.536 ± 1.415, RMSE 83.753 ± 0.924, R2 0.246 ± 0.017, SMAPE 122.555 ± 1.068
Positive Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+30min: MAE 48.482 ± 1.468, RMSE 83.634 ± 0.949, R2 0.248 ± 0.017, SMAPE 122.551 ± 1.018
System Direction (kWh)_t+30min: MAE 13378.157 ± 241.680, RMSE 19691.836 ± 351.407, R2 0.484 ± 0.019, SMAPE 117.738 ± 1.445
Negative Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+45min: MAE 51.958 ± 1.715, RMSE 88.498 ± 1.074, R2 0.158 ± 0.0